# txtai Market Research Platform - End-to-End Demo

This notebook demonstrates the full pipeline using **Apple Inc. (AAPL)** as an example:

1. **Data Ingestion**: Fetch news, SEC filings, web content, and social media
2. **Embedding & Indexing**: Chunk and embed documents into the vector index
3. **Agent Queries**: Run the orchestrator and individual agents
4. **Results Analysis**: Examine outputs and source citations

## Prerequisites

Make sure you have:
- OpenAI API key set in environment
- Optional: NewsAPI, Alpha Vantage, Reddit API keys

```bash
export OPENAI_API_KEY=sk-...
```

## Step 1: Setup and Configuration

In [ ]:
# Import required modules
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Verify API key is set
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("Warning: OPENAI_API_KEY not set. Some features may not work.")
else:
    print(f"OpenAI API key configured: {api_key[:10]}...")

print("\nSetup complete!")

## Step 2: Data Ingestion

Run the full ingestion pipeline for AAPL. This will:
- Fetch news headlines from NewsAPI/Alpha Vantage
- Pull SEC filings (10-K, 8-K, DEF 14A)
- Scrape investor relations press releases
- Fetch social media posts from Reddit and StockTwits
- Chunk documents to <=512 tokens
- Embed with sentence-transformers
- Index in SQLite-backed EmbeddingsIndex

In [ ]:
from app.pipeline.ingest import ingest_all

# Run ingestion for AAPL
print("Starting data ingestion for AAPL...")
print("=" * 50)

results = ingest_all("AAPL")

print("\n" + "=" * 50)
print("Ingestion Summary:")
print("-" * 30)
for source, count in results.items():
    print(f"  {source}: {count} documents")
print("-" * 30)
print(f"  Total: {sum(results.values())} documents indexed")

## Step 3: Verify Index Contents

Let's verify the index was created and search for some sample documents.

In [ ]:
from app.pipeline.embeddings import search, get_embeddings

# Check index exists
embeddings = get_embeddings()
print(f"Index loaded from: {embeddings.path}")

# Search for Apple-related content
print("\nSearching for 'Apple revenue'...")
results = search("Apple revenue", limit=3)

for i, result in enumerate(results, 1):
    metadata = result.get("metadata", {})
    print(f"\n--- Result {i} ---")
    print(f"Source: {metadata.get('source', 'unknown')}")
    print(f"Text: {result.get('text', '')[:200]}...")

## Step 4: Run Individual Agents

Now let's run each sub-agent to see their specialized analyses.

In [ ]:
from app.agents.sentiment import run as run_sentiment
from app.agents.diligence import run as run_diligence
from app.agents.earnings import run as run_earnings
from app.agents.regulatory import run as run_regulatory
from app.agents.web_research import run as run_web_research

# Define our query context
ticker = "AAPL"
context = {"ticker": ticker}

### 4.1: Sentiment Agent

In [ ]:
print("=" * 50)
print("SENTIMENT ANALYSIS")
print("=" * 50)

sentiment_result = run_sentiment(
    f"What is the market sentiment for {ticker}?",
    context=context
)

print(f"\nSentiment: {sentiment_result['sentiment']}")
print(f"Score: {sentiment_result['sentiment_score']:.2f} (0=bearish, 1=bullish)")
print(f"\nKey Themes:")
for theme in sentiment_result.get('key_themes', [])[:5]:
    print(f"  - {theme}")

print(f"\nSummary: {sentiment_result['summary']}")

### 4.2: Due Diligence Agent

In [ ]:
print("=" * 50)
print("DUE DILIGENCE ANALYSIS")
print("=" * 50)

diligence_result = run_diligence(
    f"What are the key risks for {ticker}?",
    context=context
)

print(f"\nRisk Flags:")
for risk in diligence_result.get('risk_flags', [])[:5]:
    severity = risk.get('severity', 'unknown')
    category = risk.get('category', 'unknown')
    desc = risk.get('description', 'N/A')[:100]
    print(f"  [{severity.upper()}] {category}: {desc}")

print(f"\nFinancial Health: {diligence_result.get('financial_health', 'N/A')}")
print(f"\nSummary: {diligence_result['summary']}")

### 4.3: Earnings Agent

In [ ]:
print("=" * 50)
print("EARNINGS ANALYSIS")
print("=" * 50)

earnings_result = run_earnings(
    f"How did {ticker}'s latest earnings go?",
    context=context
)

print(f"\nEarnings Summary: {earnings_result.get('earnings_summary', 'N/A')}")

print(f"\nKPI Trends:")
for kpi in earnings_result.get('kpi_trends', [])[:5]:
    metric = kpi.get('metric', 'N/A')
    current = kpi.get('current_value', 'N/A')
    prior = kpi.get('prior_value', 'N/A')
    change = kpi.get('change', 'N/A')
    print(f"  {metric}: {prior} -> {current} ({change})")

print(f"\nManagement Tone: {earnings_result.get('management_tone', 'N/A')[:150]}...")
print(f"\nSummary: {earnings_result['summary']}")

### 4.4: Regulatory Agent

In [ ]:
print("=" * 50)
print("REGULATORY ANALYSIS")
print("=" * 50)

regulatory_result = run_regulatory(
    f"Any regulatory issues for {ticker}?",
    context=context
)

print(f"\nFlagged Filings: {len(regulatory_result.get('flagged_filings', []))}")
for filing in regulatory_result.get('flagged_filings', [])[:3]:
    print(f"  - {filing.get('filing_type', 'N/A')}: {filing.get('issue', 'N/A')[:80]}")

print(f"\nEnforcement Signals: {len(regulatory_result.get('enforcement_signals', []))}")
print(f"Risk Assessment: {regulatory_result.get('risk_assessment', 'N/A')}")
print(f"\nSummary: {regulatory_result['summary']}")

## Step 5: Run the Orchestrator

The orchestrator routes queries to the appropriate sub-agents and synthesizes results.

In [ ]:
from app.agents.orchestrator import run as run_orchestrator

print("=" * 50)
print("ORCHESTRATOR QUERY")
print("=" * 50)

# Ask a comprehensive question
query = f"Provide a comprehensive analysis of {ticker} including sentiment, risks, and recent performance"
print(f"\nQuery: {query}\n")

orchestrator_result = run_orchestrator(query, context=context)

print(f"Response:\n{orchestrator_result.get('response', 'N/A')[:1000]}...")

print(f"\nAgents Used: {orchestrator_result.get('agents_used', [])}")
print(f"Sources Cited: {len(orchestrator_result.get('sources', []))}")

## Step 6: Metadata Filtering Demo

Demonstrate how each agent filters by source type.

In [ ]:
print("Metadata Filtering Demo")
print("=" * 50)

# Search with different source filters
sources = ["news", "sec", "web", "social"]

for source in sources:
    results = search("Apple", source_filter=source, limit=2)
    print(f"\n{source.upper()} results: {len(results)} documents")
    for r in results[:1]:
        meta = r.get('metadata', {})
        title = meta.get('title', meta.get('form_type', 'N/A'))
        print(f"  Sample: {title}")

## Summary

This notebook demonstrated the full txtai market research pipeline:

1. **Data Ingestion**: Fetched documents from multiple sources (news, SEC, web, social)
2. **Embedding & Indexing**: Chunked and embedded documents into a SQLite-backed vector index
3. **Agent Queries**: Ran individual agents (sentiment, diligence, earnings, regulatory, web research)
4. **Orchestrator**: Demonstrated multi-agent routing and synthesis
5. **Metadata Filtering**: Showed how agents filter by source type

### Key txtai Features Used

- `txtai.Embeddings`: Vector indexing with SQLite backend
- `txtai.Pipeline`: Document chunking
- `txtai.LLM`: Unified LLM interface (OpenAI + HuggingFace)
- `txtai.Agent`: Agent orchestration with tool calling
- SQL-like filtering: `WHERE tags = 'news'` for metadata filtering

### Next Steps

- Run the Streamlit app: `streamlit run app/main.py`
- Add more data sources to the ingestion pipeline
- Customize agent system prompts for your use case
- Deploy to Hugging Face Spaces (see README.md)